In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from neo4j import GraphDatabase, Result
from tqdm import tqdm
from typing import Dict, Any
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector

import pandas as pd

In [3]:
# user defined imports
from prompts import *
import helpers

In [4]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

## Parameters

In [5]:
path = "~/kg_aug_causal_disc_exp"

## Setting Up Graph

In [6]:
from config import NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD,NEO4J_DATABASE, DIRECTORY

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD),database=NEO4J_DATABASE)

def db_query(cypher: str, params: Dict[str, Any] = {}) -> pd.DataFrame:
    """Executes a Cypher statement and returns a DataFrame"""
    return driver.execute_query(
        cypher, parameters_=params, result_transformer_=Result.to_df
    )

In [7]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    refresh_schema=False,
    driver_config={"notifications_disabled_classifications": ["DEPRECATION"]}
)

/tmp/ipykernel_916031/1094186590.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(
/shared/graphrag/lib/python3.10/site-packages/langchain_community/graphs/neo4j_graph.py:404: PreviewWarning: notifications_disabled_classifications is a preview feature. It might be changed without following the deprecation policy. See also https://github.com/neo4j/neo4j-python-driver/wiki/preview-features.
  self._driver = neo4j.GraphDatabase.driver(


## Setting up LLM

In [8]:
from pydantic import BaseModel, Field
from typing import List

class Reasoning_Step(BaseModel):
    reasoning_step: str = Field(..., description="An intermediate reasoning step for breaking down the given context and query")

class Answer(BaseModel):
    reasoning: List[Reasoning_Step] = Field(..., description="List of reasoning steps")
    conclusion: bool = Field(..., description="The culminating final conclusion or answer to the question")

In [9]:
from llm_client import get_client
generator = get_client(schema=Answer)

# Local Retriever

In [10]:
# parameters for the local search query
from config import topChunks, topCommunities, topRels, topEntities

lc_retrieval_query = helpers.load_query("local_search.cypher")
kw_retrieval_query = helpers.load_query("keyword_search.cypher")

In [11]:
# variables of interest
with open("variable_definitions/default_definitions.json", "r") as file:
    def_map = json.load(file)

In [12]:
db_query(
    """
    CREATE VECTOR INDEX vector IF NOT EXISTS
    FOR (n:__Entity__)
    ON n.embedding
    OPTIONS {indexConfig: {
      `vector.dimensions`: 768,
      `vector.similarity_function`: "cosine"
    }};
    """
)

""


In [13]:
db_query("SHOW INDEXES")

,id,name,state,populationPercent,type,entityType,labelsOrTypes,properties,indexProvider,owningConstraint,lastRead,readCount
0,8,constraint_8be13a40,ONLINE,100.0,RANGE,NODE,[__Community__],[id],range-1.0,constraint_8be13a40,2025-12-05T20:27:49.504000000+00:00,220440
1,1,constraint_907a464e,ONLINE,100.0,RANGE,NODE,[__Entity__],[id],range-1.0,constraint_907a464e,2025-12-04T22:30:49.273000000+00:00,132894
2,6,vector,ONLINE,100.0,VECTOR,NODE,[__Entity__],[embedding],vector-1.0,None,2025-12-09T18:31:48.014000000+00:00,4364


In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

embedding = HuggingFaceEmbeddings(
    model_name="pritamdeka/S-PubMedBert-MS-MARCO",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

/tmp/ipykernel_916031/2722532535.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


In [15]:
lc_retrieval_query = helpers.load_query("local_search.cypher")
lc_vector = Neo4jVector.from_existing_index(
    embedding=embedding,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    index_name="vector", # may need to alter
    search_type="vector",
    # keyword_index_name="keyword",
    retrieval_query=lc_retrieval_query,
)

In [16]:
from helpers import *
from build_context import stringify_report, format_triplet, construct_query_context
from config import query_context_window

def retrieve_context_query(query) -> str:

    res = lc_vector.similarity_search(
        query,
        k=topEntities,
        params={
            "topChunks": topChunks,
            "topCommunities": topCommunities,
            "topRels": topRels
        }
    )

    metadata = res[0].metadata
    reports = [stringify_report(report) for report in metadata["Reports"]]
    chunks = metadata["Chunks"]
    relationships = [format_triplet(triplet) for triplet in metadata["Relationships"]]

    return construct_query_context(relationships, chunks, reports, max_context_window=query_context_window)

In [17]:
def local_retriever(query, var1, var2, summary, debug=False):
    if debug:
        print(reduce(query, var1, var2, summary, def_map))
    response = generator(reduce(query, var1, var2, summary, def_map), sampling_params={"n":1, "temperature":0.0, "top_k":1})

    return response.conclusion, helpers.reasoning_to_string(response)
    

In [18]:
def llm_retriever(query, var1, var2, debug=False):
    if debug:
        print(predict(query, var1, var2, def_map))
    response = generator(predict(query, var1, var2, def_map), sampling_params={"n":1, "temperature":0.0, "top_k":1})
    return response.conclusion, helpers.reasoning_to_string(response)


In [19]:
from promptsd.query_prompts import plausibility_prompt, temporality_prompt, causal_lit_prompt, association_prompt

def query_local_causality(row):
    var1, var2, label = row['var1'], row['var2'], row["label"]
    # bandaid for now
    var1 = "Sleep disturbance" if var1 == "Sleep" else var1
    var2 = "Sleep disturbance" if var2 == "Sleep" else var2
        
    report = retrieve_context_query(var1, var2)
    pquery = plausibility_prompt(var1, var2)
    aquery = association_prompt(var1, var2)
    tquery = temporality_prompt(var1, var2)
    plausibility, preasoning = local_retriever(pquery, var1, var2, report)
    association, areasoning = local_retriever(aquery, var1, var2, report)
    temporality, treasoning = local_retriever(tquery, var1, var2, report)
    return [var1, var2, plausibility, preasoning, association, areasoning, temporality, treasoning, report, label]

In [20]:
def query_llm_causality(row):
    var1, var2, label = row['var1'], row['var2'], row["label"]
    # bandaid for now
    var1 = "Sleep disturbance" if var1 == "Sleep" else var1
    var2 = "Sleep disturbance" if var2 == "Sleep" else var2
        
    pquery = plausibility_prompt(var1, var2)
    aquery = association_prompt(var1, var2)
    tquery = temporality_prompt(var1, var2)
    plausibility, preasoning = llm_retriever(pquery, var1, var2)
    association, areasoning = llm_retriever(aquery, var1, var2)
    temporality, treasoning = llm_retriever(tquery, var1, var2)
    return [var1, var2, plausibility, preasoning, association, areasoning, temporality, treasoning, label]

In [21]:
full = pd.read_csv(f"{path}/data/full_cleaned.csv").drop(columns=["Unnamed: 0"])

# Experiments

## LLM

In [ ]:
from prompts import *
res = full.apply(query_llm_causality, axis=1)

In [ ]:
columns = "Var1", "Var2", "Plausibility", "Plausibility Reasoning", "Association", "Association Reasoning", "Temporality", "Temporality Reasoning", "Label"
llm_res = pd.DataFrame(res.to_list(), columns=columns)
llm_res.to_csv("results/llm.csv")
llm_res

In [ ]:
from sklearn.metrics import f1_score
print("RESULTS FOR LLM ONLY")
print(f1_score(llm_res["Label"], llm_res["Plausibility"]))

In [ ]:
print(llm_res["Plausibility"].value_counts())

## Local Search

In [173]:
from prompts import *
res = full.apply(query_local_causality, axis=1)

NameError: name 'full' is not defined

In [ ]:
columns = "Var1", "Var2", "Plausibility", "Plausibility Reasoning", "Association", "Association Reasoning", "Temporality", "Temporality Reasoning", "Report", "Label"
local_res = pd.DataFrame(res.to_list(), columns=columns)
local_res.to_csv("results/kgrag.csv")
local_res

In [ ]:
from sklearn.metrics import f1_score
print("RESULTS FOR LOCAL")
print(f1_score(local_res["Label"], local_res["Plausibility"]))

In [ ]:
print(local_res["Plausibility"].value_counts())

# Toy Example

In [44]:
from helpers import *
from build_context import stringify_report, format_triplet, construct_query_context
from config import query_context_window, topEntities

def retrieve_context_query(query) -> str:

    res = lc_vector.similarity_search(
        query,
        k=topEntities,
        params={
            "topChunks": topChunks,
            "topCommunities": topCommunities,
            "topRels": topRels
        }
    )

    metadata = res[0].metadata
    print(metadata["Reports"])
    reports = [stringify_report(report) for report in metadata["Reports"]]
    chunks = metadata["Chunks"]
    relationships = [format_triplet(triplet) for triplet in metadata["Relationships"]]

    return construct_query_context(relationships, chunks, reports, max_context_window=query_context_window)

In [45]:
lc_retrieval_query = helpers.load_query("local_search.cypher")
lc_vector = Neo4jVector.from_existing_index(
    embedding=embedding,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    index_name="vector", # may need to alter
    search_type="vector",
    # keyword_index_name="keyword",
    retrieval_query=lc_retrieval_query,
)

In [46]:
def local_retriever(query, var1, var2, summary, debug=False):
    if debug:
        print(reduce(query, var1, var2, summary, def_map))
    response = generator(reduce(query, var1, var2, summary, def_map), sampling_params={"n":1, "temperature":0.0, "top_k":1})

    return response.conclusion, helpers.reasoning_to_string(response)

In [71]:
from promptsd.query_prompts import plausibility_prompt, temporality_prompt, association_prompt
from prompts_directions import DIRECT_CAUSAL_PROMPT
from prompts import reduce


var1, var2 = "Alcohol", "Financial_level"
query = association_prompt(var1, var2)
report = retrieve_context_query(f"{def_map.get(var1)} {def_map.get(var2)}")
answer = local_retriever(query, var1, var2, report, debug=True)

[{'detailed_findings': ['Alcohol consumption may be associated with sleep disturbances: The text suggests that alcohol consumption may be associated with various sleep disturbances, including insomnia, obstructive sleep apnea, circadian abnormalities, short sleep duration, and poor sleep quality. Alcohol consumption, particularly binge drinking, has been linked to adverse health effects and sleep problems.', 'Alcohol consumption may affect fear avoidance beliefs: The text indicates that alcohol consumption may affect fear avoidance beliefs, potentially contributing to increased fear avoidance. This may be due to impaired judgment or increased risk-taking associated with alcohol use.', 'Alcohol consumption may be associated with financial level: The text suggests that there may be an association between financial level and alcohol consumption, as financial constraints can affect the ability to afford alcohol. However, the relationship is not explicitly stated, and it is possible that fi

In [72]:
print(answer[1])

Reasoning Process:
Step 1: The report provides several relationships between Alcohol and Financial_level, including 'Alcohol consumption may be associated with financial level', 'Alcohol consumption may affect financial level', 'There may be an association between financial level and alcohol consumption', and 'Financial level may influence alcohol consumption'. These relationships suggest a statistical dependency between the two variables.
Step 2: The absence of a relationship does not necessarily mean that there is no association. However, in this case, the multiple relationships provided in the report provide strong evidence of an association between Alcohol and Financial_level.

Conclusion: Yes


In [64]:
import helpers
helpers.token_count(report)

7993

In [55]:
print(report)

# Report:
## Chunks:

 al.,](#page-7-3) [2012\)](#page-7-3), and *moderate drinkers* (34.8%; defined as *current drinkers* who that have less than or equal to 210 g of ethanol per week). #### Genetic Instruments There are two genetic variants commonly used in MR studies of alcohol use: the alcohol dehydrogenase 1B gene (*ADH1B* rs1229984) and the aldehyde dehydrogenase 2 gene (*ALDH2* rs671), both of which encode enzymes involved in the metabolic pathway for ethanol and can change the metabolic balance of acetaldehyde in human body ([Peng and Yin, 2009\)](#page-7-5). In the human body, ethanol is first converted to acetaldehyde by alcohol dehydrogenase (ADH) and then to acetate by aldehyde dehydrogenase (ALDH). The enzyme activity of ADH and ALDH are largely determined by the number of effect alleles (i.e., A-allele) in both *ADH1B* rs1229984 and *ALDH2* rs671. In East Asian populations, *ALDH2* rs671 alleles exist with three genotypes, GG (# of A allele = 0), AG (# of A allele = 1), a

In [56]:
print(report)

# Report:
## Chunks:

 al.,](#page-7-3) [2012\)](#page-7-3), and *moderate drinkers* (34.8%; defined as *current drinkers* who that have less than or equal to 210 g of ethanol per week). #### Genetic Instruments There are two genetic variants commonly used in MR studies of alcohol use: the alcohol dehydrogenase 1B gene (*ADH1B* rs1229984) and the aldehyde dehydrogenase 2 gene (*ALDH2* rs671), both of which encode enzymes involved in the metabolic pathway for ethanol and can change the metabolic balance of acetaldehyde in human body ([Peng and Yin, 2009\)](#page-7-5). In the human body, ethanol is first converted to acetaldehyde by alcohol dehydrogenase (ADH) and then to acetate by aldehyde dehydrogenase (ALDH). The enzyme activity of ADH and ALDH are largely determined by the number of effect alleles (i.e., A-allele) in both *ADH1B* rs1229984 and *ALDH2* rs671. In East Asian populations, *ALDH2* rs671 alleles exist with three genotypes, GG (# of A allele = 0), AG (# of A allele = 1), a

# Saving the Prompts

In [ ]:
import json

# variables of interest
with open("variable_definitions/default_definitions.json", "r") as file:
    def_map = json.load(file)

In [ ]:
from promptsd.query_prompts import plausibility_prompt, temporality_prompt, causal_lit_prompt, association_prompt

var1 = "Sex"
var2 = "Anxiety"

with open("raw_prompts/plausibility_prompt.txt", "w") as f:
    print(plausibility_prompt(var1, var2), file=f)

with open("raw_prompts/temporality_prompt.txt", "w") as f:
    print(temporality_prompt(var1, var2), file=f)

with open("raw_prompts/causal_lit_prompt.txt", "w") as f:
    print(causal_lit_prompt(var1, var2), file=f)

with open("raw_prompts/association_prompt.txt", "w") as f:
    print(association_prompt(var1, var2), file=f)

In [ ]:
pquery = plausibility_prompt(var1, var2)
aquery = association_prompt(var1, var2)
tquery = temporality_prompt(var1, var2)

In [ ]:
from prompts import *
with open("raw_prompts/kg+llm_prompt_plausibility.txt", "w") as f:
    print(reduce(pquery, var1, var2, "", def_map), file=f)

with open("raw_prompts/kg+llm_prompt_association.txt", "w") as f:
    print(reduce(aquery, var1, var2, "", def_map), file=f)

with open("raw_prompts/kg+llm_prompt_temporality.txt", "w") as f:
    print(reduce(tquery, var1, var2, "", def_map), file=f)

In [ ]:
with open("raw_prompts/llm+rag_prompt_plausibility.txt", "w") as f:
    print(reduce_rag(pquery, var1, var2, "", def_map), file=f)

with open("raw_prompts/llm+rag_prompt_association.txt", "w") as f:
    print(reduce_rag(aquery, var1, var2, "", def_map), file=f)

with open("raw_prompts/llm+rag_prompt_temporality.txt", "w") as f:
    print(reduce_rag(tquery, var1, var2, "", def_map), file=f)

In [ ]:
with open("raw_prompts/llm_prompt_plausibility.txt", "w") as f:
    print(predict(pquery, var1, var2,  def_map), file=f)

with open("raw_prompts/llm_prompt_association.txt", "w") as f:
    print(predict(aquery, var1, var2,  def_map), file=f)

with open("raw_prompts/llm_prompt_temporality.txt", "w") as f:
    print(predict(tquery, var1, var2,  def_map), file=f)